# Round mosaics — live quick look at every round

Builds a mosaic (one per real imaging color) for EVERY round in
`round_info.csv`, from a single frame per FOV near `TARGET_Z_UM` -- no full
z-stack read, so it's light enough to run continuously alongside a real
acquisition (reading straight off the NAS while HAL/Dave is still writing).
Flat-field correction is OFF by default (fastest option) but can be turned
on (`ENABLE_FFC`, section 4) -- see that section for what it costs. This is
a quick-look tool, not a replacement for `analysis/02_round_scheduler.ipynb`'s
production mosaics (mid-z, optional FFC, built only once a round is 100%
done) -- both can run at the same time without conflicting; this notebook
saves to `SAMPLE_DIR/figures/`, not `analysis/mosaics/`.

**What it does**: a one-time catch-up pass (section 5) mosaics every round
that's ALREADY fully imaged when you start the notebook -- e.g. if you start
this mid-experiment (say, during round 6), rounds 1-5 get built first, in
order, before anything else happens. The live loop (section 6) then watches
whichever round is currently being imaged (auto-detected, same logic as
`imaged_fovs.ipynb`), reading any newly-appeared FOV's single frame per
color and redrawing that round's mosaic(s) so far. Once a round finishes,
its final mosaic is saved and the notebook automatically moves on to
watching the next round -- this is meant to be started once and left
running for the whole experiment, not just one round.

**Reading from the NAS while it's being written to**: this notebook's reads
and HAL's writes share the same underlying disk/network link, so there's a
real, if usually small, risk of one slowing the other down -- see section
5's `CATCHUP_READ_DELAY_SEC` for the one place this notebook does a
sustained read burst, and the rationale in its comment for how that default
was chosen (also explained in the chat that added it).

**Usage**: run every cell once, then run the last cell and leave it running.
Interrupt the kernel to stop at any time -- whatever's been built so far is
already saved.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy.spatial import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.acquisition.configs    import find_frame_table_for_hal_config
from MERci.acquisition.positions  import find_exterior_fovs
from MERci.analysis.fov           import create_thumbnail
from MERci.analysis.round         import create_mosaic, create_mosaic_ffc
from MERci.analysis.ffc           import (
    compute_ffc_field_for_color, apply_ffc, save_ffc_field, load_ffc_field,
)
from MERci.scheduler              import resolve_round_flip_y

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "round_mosaics"   # used to namespace this notebook's cache + figure files

# Real stage z (um) to build every round's mosaic at -- for each round/color,
# the frame whose own z is closest to this gets used. One frame per FOV per
# color, no z-stack -- deliberately light enough to run continuously during a
# real acquisition. Change and re-run section 3 to pick a different depth
# (e.g. to match where your tissue actually has signal).
TARGET_Z_UM = 10.0

# On-demand mosaic for ONE specific round (section 7), independent of the
# catch-up pass/live loop above -- for a quick look at a particular round
# (e.g. the cells round, usually the most informative single overview of the
# tissue) without waiting for it to come up naturally. None = "cells" (the
# real name in round_info.csv), matching ROUND_IMAGING_TYPE's own convention
# in correct_camera_rotation.ipynb -- resolves to whichever round has a
# series with that imaging_type. Set to an int instead (a real imaging_round
# number) to pick a round directly, bypassing the imaging_type lookup. None
# disables section 7 entirely.
SPECIFIC_ROUND = "cells"

# Flat-field correction -- off by default (fastest option, no extra reads).
# Turning it on costs FFC_N_FOVS extra one-time reads PER COLOR (not per
# round -- vignetting is a fixed optical property, cached to disk after the
# first computation) -- see section 4.
ENABLE_FFC = False
FFC_N_FOVS = 10

POLL_INTERVAL_SEC = 5         # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 24*60*7  # safety cap -- this notebook is meant to run for a
                               # whole multi-round experiment, not just one round

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

THUMBNAILS_DIR = config.analysis_dir / "thumbnails"   # shared with 01_fov_scheduler.ipynb's own convention
THUMBNAILS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME   # FFC fields only (NOTEBOOK_GUIDELINES.md #2)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Deliberately SAMPLE_DIR/figures/, not analysis/mosaics/ -- see markdown
# above: this is a quick-look tool, kept out of the way of the production
# mosaics analysis/02_round_scheduler.ipynb builds at the same filenames.
FIGURES_DIR = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"TARGET_Z_UM  : {TARGET_Z_UM}")
print(f"ENABLE_FFC   : {ENABLE_FFC}")

## 3 — Resolve each round's real colors -> nearest-z frame index

Reads each round's own frame table (via its HAL config, same resolution
`prepare_imaging` already uses) and picks, per real (non-blank) color, the
frame whose actual `z` (um) is closest to `TARGET_Z_UM`. Rounds sharing the
same HAL config (common for repeated bits rounds) resolve identically, but
each round is still looked up independently since nothing guarantees that in
general.

In [ ]:
def resolve_round_color_frames(round_id):
    # {color_nm: frame_idx} for round_id's own frame table -- the frame
    # closest to TARGET_Z_UM for every real (non-blank) color it has.
    color_frames = {}
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        frame_table_path = find_frame_table_for_hal_config(
            config.settings_dir / s.hal_config, config.metadata_dir)
        if frame_table_path is None:
            continue
        frame_table = pd.read_csv(frame_table_path)
        for color in sorted(frame_table["color"].dropna().unique()):
            candidates = frame_table[frame_table["color"].round(0) == round(color)]
            frame_idx = int((candidates["z"] - TARGET_Z_UM).abs().idxmin())
            resolved_z = float(candidates.loc[frame_idx, "z"])
            color_frames[float(color)] = frame_idx
            if abs(resolved_z - TARGET_Z_UM) > 5.0:
                print(f"  round {round_id}, color {color:.0f} nm: nearest available z is "
                      f"{resolved_z:.1f} um (requested {TARGET_Z_UM:.1f} um) -- frame {frame_idx}")
    return color_frames


ROUND_COLOR_FRAMES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frames(round_id)
    if cf:
        ROUND_COLOR_FRAMES[round_id] = cf
    print(f"Round {round_id}: colors {sorted(cf)} -> frame indices {cf}")

## 4 — Flat-field correction (optional)

Off by default (`ENABLE_FFC = False` in section 2) -- every mosaic below is
then built the fast way, one independently-contrast-stretched thumbnail per
FOV. Turning it on divides out a per-color illumination/vignette field
before thumbnailing, at the cost of `FFC_N_FOVS` (default 10) extra one-time
raw-frame reads PER COLOR -- small and one-off, since vignetting is a fixed
property of the microscope/channel, not of any one round, so the same field
is reused for every round afterward and cached to disk
(`analysis/cache/round_mosaics/ffc_field_{color}nm.npz`).

Samples come from real exterior FOVs (the outer edge of the imaged grid,
via the same `find_exterior_fovs` the production FFC pipeline uses) that are
ALREADY imaged in whatever round is being built when the field is first
needed -- not a fixed reference round, since round 1 might not be the first
one that actually has all `FFC_N_FOVS` exterior FOVs done yet.

**Known tradeoff, stated plainly**: the one-time catch-up pass (section 5)
applies FFC the same way production mosaics do -- read every FOV's real raw
frame, then ONE shared contrast stretch across the whole assembled canvas
(`create_mosaic_ffc`), which looks visibly better than independent per-tile
stretching. The LIVE loop (section 6) can't do that -- keeping every round's
full-resolution raw frames in memory for the whole time it's being imaged
isn't affordable (a real multi-thousand-FOV round's raw frames alone would
be many GB), so live mode instead divides out the FFC field frame-by-frame
right after reading it, THEN thumbnails and stretches each tile
independently, same as the non-FFC path. You'll see the flat-field
correction live, just not the extra global-contrast-stretch polish, and a
round finished by the live loop stays at that quality permanently -- section
5's "skip if the mosaic file already exists" check means simply re-running
it afterward will NOT upgrade a live-built round to the catch-up-quality
rendering. Delete that round's `round_mosaics.round{id:03d}_{color}nm.png`
file(s) first if you want section 5 to rebuild it at catch-up quality.

In [ ]:
coords_arr   = np.array([meta.fovs[f].position for f in sorted(meta.fovs)])
nn_dist, _   = KDTree(coords_arr).query(coords_arr, k=2)
STEP_SIZE_UM = float(np.median(nn_dist[:, 1]))
EXTERIOR_FOV_IDS = find_exterior_fovs(
    {f: meta.fovs[f].position for f in meta.fovs}, STEP_SIZE_UM)

_ffc_fields = {}   # {color_nm: np.ndarray} -- in-memory cache for this session


def get_or_compute_ffc_field(color_nm, frame_idx, available_fov_ids, series):
    if not ENABLE_FFC:
        return None
    if color_nm in _ffc_fields:
        return _ffc_fields[color_nm]

    cache_path = CACHE_DIR / f"ffc_field_{color_nm:.0f}nm.npz"
    if cache_path.exists():
        field, _ = load_ffc_field(cache_path)
        _ffc_fields[color_nm] = field
        print(f"Loaded cached FFC field ({color_nm:.0f} nm): {cache_path}")
        return field

    candidate_ids = sorted(EXTERIOR_FOV_IDS & set(available_fov_ids))[:FFC_N_FOVS]
    if not candidate_ids:
        print(f"  FFC ({color_nm:.0f} nm): no exterior FOVs available yet -- will retry later.")
        return None

    samples = []
    for fov_id in candidate_ids:
        paths = [s.resolve_path(fov_id, config.image_suffix) for s in series]
        existing = [p for p in paths if p.exists()]
        if existing:
            samples.append((existing[0], frame_idx))
    if not samples:
        return None

    field, field_meta = compute_ffc_field_for_color(
        samples, frame_width=config.frame_width, frame_height=config.frame_height)
    save_ffc_field(cache_path, field, field_meta)
    _ffc_fields[color_nm] = field
    print(f"Computed FFC field ({color_nm:.0f} nm) from {len(samples)} exterior FOV(s): {cache_path}")
    return field


print(f"STEP_SIZE_UM       : {STEP_SIZE_UM:.1f} um")
print(f"Exterior FOV count : {len(EXTERIOR_FOV_IDS)}")

## 5 — Catch-up pass: mosaic every already-finished round once

Only rounds with 100% of their FOVs already imaged, and only if that round's
mosaic file(s) don't already exist yet (so re-running this notebook after an
interruption doesn't redo finished work). Rounds are processed in order
(round 1, then 2, ...) -- if you start this notebook mid-experiment (say,
during round 6), rounds 1-5 are built here, in that order, BEFORE section
6's live loop ever starts watching round 6. The round currently being
imaged (if any) is deliberately skipped here -- section 6 handles it,
updating its mosaic as new FOVs actually appear rather than only once.

**`CATCHUP_READ_DELAY_SEC`** -- reading a whole already-finished round back
to back (up to `meta.n_fovs` raw-frame reads in a tight loop) is the only
SUSTAINED read burst this notebook does; the live loop's own reads are
already naturally paced by "wait for new files to appear," which can't
outrun HAL's own writing. This burst, though, competes with round 6's
ACTIVE writes for the same disk/network link at the exact moment you start
the notebook. There's no universal safe number here (it depends on this
NAS's real hardware/network capacity and how much other traffic already
uses it, which nothing in this notebook can measure for you) -- the default
below is a conservative, deliberately small pacing gap chosen for a
different, checkable reason: HAL's own write demand is small and steady (one
frame is `image_size_px**2 * 2` bytes -- e.g. ~8 MB for a 2048x2048 uint16
frame -- written roughly once per `exposure_time`, typically a few hundred
ms, i.e. on the order of tens of MB/s), well under what even a modest 1 GbE
link can carry (~118 MB/s) on bytes alone -- so the real risk isn't raw
throughput, it's IOPS/seek contention from many small, scattered file reads
landing on the same storage the sequential HAL write stream is using,
which byte-counting alone doesn't capture. `CATCHUP_READ_DELAY_SEC=0.02`
(20 ms) keeps this notebook's own read rate capped at ~50 files/sec
regardless of how fast the NAS could otherwise serve them, which is slow
enough to leave real headroom without making the catch-up pass
impractically slow (about 20s of pure pacing per 1000-FOV round). Treat this
as a starting point, not a proof of safety -- if you suspect it's still
affecting acquisition, check whether HAL's own per-frame write timing
visibly changes while this cell runs (real file-write-mtime gaps, the same
technique `misc/measure_tissue_thickness_test.ipynb` section 8 already
uses) and raise the delay if so.

In [ ]:
CATCHUP_READ_DELAY_SEC = 0.02   # see markdown above for the rationale


def round_imaged_fov_ids(round_id):
    series = meta.series_for_round(round_id)
    return [
        fov_id for fov_id in sorted(meta.fovs)
        if any(s.resolve_path(fov_id, config.image_suffix).exists() for s in series)
    ]


def round_n_imaged(round_id):
    return len(round_imaged_fov_ids(round_id))


def mosaic_path(round_id, color_nm):
    return FIGURES_DIR / f"{NOTEBOOK_NAME}.round{round_id:03d}_{color_nm:.0f}nm.png"


def build_round_mosaic(round_id, color_frames, fov_ids):
    # Read one frame per fov_id per color (paced by CATCHUP_READ_DELAY_SEC),
    # and save/return one mosaic per color -- FFC-corrected (whole-canvas
    # shared contrast stretch) if ENABLE_FFC, else independently-stretched
    # thumbnails (cached to THUMBNAILS_DIR, shared with 01_fov_scheduler.
    # ipynb's own convention). Returns {color_nm: canvas}.
    series  = meta.series_for_round(round_id)
    flip_y  = resolve_round_flip_y(round_id, config, meta)
    canvases = {}
    for color_nm, frame_idx in color_frames.items():
        ffc_field = get_or_compute_ffc_field(color_nm, frame_idx, fov_ids, series)
        thumbnails, raw_frames, positions = {}, {}, {}
        for fov_id in fov_ids:
            existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
            existing = [p for p in existing if p.exists()]
            if not existing:
                continue
            image_path = existing[0]
            if ffc_field is not None:
                raw_frames[fov_id] = read_image_frames(
                    image_path, [frame_idx],
                    frame_width=config.frame_width, frame_height=config.frame_height,
                )[0]
            else:
                thumb_path = THUMBNAILS_DIR / f"{image_path.stem}_frame{frame_idx:03d}.png"
                if thumb_path.exists():
                    from PIL import Image
                    thumb = np.array(Image.open(str(thumb_path)))
                else:
                    frame = read_image_frames(
                        image_path, [frame_idx],
                        frame_width=config.frame_width, frame_height=config.frame_height,
                    )[0]
                    thumb = create_thumbnail(frame, thumb_path, target_size=config.thumbnail_size,
                                              percentile_clip=config.thumbnail_percentile_clip)
                thumbnails[fov_id] = thumb
            positions[fov_id] = meta.fovs[fov_id].position
            time.sleep(CATCHUP_READ_DELAY_SEC)

        if ffc_field is not None and raw_frames:
            canvases[color_nm] = create_mosaic_ffc(
                raw_frames, positions, mosaic_path(round_id, color_nm),
                ffc_field=ffc_field, crop_px=0, thumbnail_size=config.thumbnail_size,
                padding=config.mosaic_padding, flip_y=flip_y,
                percentile_clip=config.thumbnail_percentile_clip,
            )
        elif thumbnails:
            canvases[color_nm] = create_mosaic(
                thumbnails, positions, mosaic_path(round_id, color_nm),
                thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding, flip_y=flip_y,
            )
    return canvases


for round_id, color_frames in ROUND_COLOR_FRAMES.items():
    if round_n_imaged(round_id) < meta.n_fovs:
        continue   # not finished -- section 6's live loop will pick it up when it's the active round
    if all(mosaic_path(round_id, c).exists() for c in color_frames):
        continue   # already built by a previous run of this notebook
    print(f"Catch-up: building mosaic(s) for already-finished round {round_id}...")
    build_round_mosaic(round_id, color_frames, sorted(meta.fovs))
print("Catch-up pass done.")

## 6 — Live loop: watch the active round, advance automatically as rounds finish

Same round auto-detection as `imaged_fovs.ipynb` (prefers a round with SOME
but not ALL FOVs imaged; during a fluidics gap, points at the round after the
most recently completed one). Unlike `imaged_fovs.ipynb`, this loop keeps
running across MULTIPLE rounds for the whole experiment -- once the round
it's watching finishes, it saves that round's final mosaic, frees its
in-memory thumbnails, and moves on to whichever round is next.

If `ENABLE_FFC`, each newly-read frame is divided by that color's FFC field
(computed/cached in section 4, from whichever round first has enough
exterior FOVs done) before thumbnailing -- see section 4's markdown for why
this is a lighter-weight FFC treatment than the catch-up pass's.

Interrupt the kernel to stop -- whatever's been built so far is already
saved to `figures/`.

In [ ]:
def detect_active_round():
    best_round, best_latest, best_in_progress = None, -1.0, False
    for round_id in ROUND_COLOR_FRAMES:
        n_imaged = round_n_imaged(round_id)
        if n_imaged == 0:
            continue
        in_progress = n_imaged < meta.n_fovs
        # No real mtime scan here (unlike imaged_fovs.ipynb) -- n_imaged alone
        # is enough to rank candidates since round ids already have a real
        # chronological order (rounds are imaged strictly in round_id order).
        latest = float(round_id)
        if (in_progress, latest) > (best_in_progress, best_latest):
            best_round, best_latest, best_in_progress = round_id, latest, in_progress

    if best_round is None:
        return sorted(ROUND_COLOR_FRAMES)[0]

    if not best_in_progress:
        round_ids = sorted(ROUND_COLOR_FRAMES)
        idx = round_ids.index(best_round)
        if idx + 1 < len(round_ids):
            return round_ids[idx + 1]

    return best_round


watched_round   = None
thumbnails_by_color = {}   # {color_nm: {fov_id: thumbnail}}
imaged_fov_ids  = set()

start_time = time.time()
poll_count = 0

try:
    while True:
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded.")
            break

        active_round = detect_active_round()
        if active_round != watched_round:
            watched_round = active_round
            thumbnails_by_color = {c: {} for c in ROUND_COLOR_FRAMES[watched_round]}
            imaged_fov_ids = set()
            print(f"Now watching round {watched_round} "
                  f"(colors: {sorted(ROUND_COLOR_FRAMES[watched_round])}).")

        color_frames = ROUND_COLOR_FRAMES[watched_round]
        series       = meta.series_for_round(watched_round)
        flip_y       = resolve_round_flip_y(watched_round, config, meta)

        poll_count += 1
        newly_imaged = []
        for fov_id in sorted(meta.fovs):
            if fov_id in imaged_fov_ids:
                continue
            existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
            existing = [p for p in existing if p.exists()]
            if existing:
                imaged_fov_ids.add(fov_id)
                newly_imaged.append((fov_id, existing[0]))

        for fov_id, image_path in newly_imaged:
            for color_nm, frame_idx in color_frames.items():
                frame = read_image_frames(
                    image_path, [frame_idx],
                    frame_width=config.frame_width, frame_height=config.frame_height,
                )[0]
                ffc_field = get_or_compute_ffc_field(color_nm, frame_idx, imaged_fov_ids, series)
                if ffc_field is not None:
                    frame = apply_ffc(frame, ffc_field)
                thumb_path = THUMBNAILS_DIR / f"{image_path.stem}_frame{frame_idx:03d}.png"
                thumb = create_thumbnail(frame, thumb_path, target_size=config.thumbnail_size,
                                          percentile_clip=config.thumbnail_percentile_clip)
                thumbnails_by_color[color_nm][fov_id] = thumb

        colors = sorted(color_frames)
        fig, axes = plt.subplots(1, len(colors), figsize=(6 * len(colors), 6), squeeze=False)
        for ax, color_nm in zip(axes[0], colors):
            thumbs = thumbnails_by_color[color_nm]
            if thumbs:
                positions = {f: meta.fovs[f].position for f in thumbs}
                canvas = create_mosaic(
                    thumbs, positions, mosaic_path(watched_round, color_nm),
                    thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding, flip_y=flip_y,
                )
                ax.imshow(canvas, cmap="gray")
            ax.set_title(f"round {watched_round} — {color_nm:.0f} nm")
            ax.axis("off")
        fig.tight_layout()

        clear_output(wait=True)
        display(fig)
        plt.close(fig)
        elapsed = time.time() - start_time
        ffc_status = ("on (" + ", ".join(f"{c:.0f}nm" for c in sorted(_ffc_fields)) + ")"
                      if ENABLE_FFC else "off")
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | round {watched_round} | "
              f"imaged {len(imaged_fov_ids)}/{meta.n_fovs} | FFC: {ffc_status} | "
              f"next check in {POLL_INTERVAL_SEC}s")
        # Any print from inside this iteration's own read/thumbnail loop above
        # (e.g. get_or_compute_ffc_field's one-time "Computed FFC field..."
        # message) gets wiped by THIS iteration's own clear_output() call
        # before it's ever visible -- folding FFC status into this persistent
        # status line (which survives until the NEXT clear_output) is how the
        # user actually gets to see it, rather than a flash-and-vanish print.
        # Once this round is fully imaged, the top-of-loop `active_round !=
        # watched_round` check will detect the switch to the next round on
        # the NEXT iteration and reset thumbnails_by_color there -- UNLESS
        # this is already the last round, which has no next round to switch
        # to (detect_active_round's own fallback just keeps returning it) --
        # stop the whole loop in that case rather than polling a finished
        # experiment forever.
        if watched_round == max(ROUND_COLOR_FRAMES) and len(imaged_fov_ids) >= meta.n_fovs:
            print("All rounds finished.")
            break

        time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user while watching round {watched_round} "
          f"({len(imaged_fov_ids)}/{meta.n_fovs} FOVs imaged).")

## 7 — On-demand: mosaic for one specific round

Independent of sections 5/6 above -- builds (or rebuilds) a mosaic for just
`SPECIFIC_ROUND` (section 2; `None` skips this section), from whatever FOVs
are imaged for it right now, whether that round is fully finished, still in
progress, or hasn't come up in the live loop yet. Useful for a quick look at
a particular round (the cells round by default -- usually the single most
informative overview of the tissue) without waiting for the catch-up pass
or live loop to reach it. Uses the same `build_round_mosaic` (and the same
`CATCHUP_READ_DELAY_SEC` pacing) as section 5's catch-up pass -- re-run this
cell any time to refresh it against whatever's been imaged since.

In [ ]:
def resolve_round_by_imaging_type(imaging_type):
    target = imaging_type.strip().lower()
    for round_id in sorted(meta.rounds):
        for s in meta.series_for_round(round_id):
            if (s.imaging_type or "").strip().lower() == target:
                return round_id
    return None


if SPECIFIC_ROUND is None:
    print("SPECIFIC_ROUND is None -- skipping.")
else:
    if isinstance(SPECIFIC_ROUND, str):
        specific_round_id = resolve_round_by_imaging_type(SPECIFIC_ROUND)
        if specific_round_id is None:
            raise ValueError(f"No round has a series with imaging_type={SPECIFIC_ROUND!r} "
                              f"-- check round_info.csv, or set SPECIFIC_ROUND to an explicit round id.")
    else:
        specific_round_id = int(SPECIFIC_ROUND)
        if specific_round_id not in ROUND_COLOR_FRAMES:
            raise ValueError(f"Round {specific_round_id} has no resolved colors "
                              f"(see section 3) -- check it's a real imaging_round in round_info.csv.")

    fov_ids = round_imaged_fov_ids(specific_round_id)
    print(f"Round {specific_round_id}: {len(fov_ids)}/{meta.n_fovs} FOV(s) imaged so far.")

    if not fov_ids:
        print("No FOVs imaged yet for this round -- nothing to mosaic.")
    else:
        canvases = build_round_mosaic(specific_round_id, ROUND_COLOR_FRAMES[specific_round_id], fov_ids)
        colors = sorted(canvases)
        fig, axes = plt.subplots(1, max(len(colors), 1), figsize=(6 * max(len(colors), 1), 6), squeeze=False)
        for ax, color_nm in zip(axes[0], colors):
            ax.imshow(canvases[color_nm], cmap="gray")
            ax.set_title(f"round {specific_round_id} — {color_nm:.0f} nm")
            ax.axis("off")
        fig.tight_layout()
        plt.show()